# FedASMU: dynamic staleness-aware model update

Asynchronous FL in which **both sides adapt to staleness**: the server mixes each arriving model in with a weight that it learns per client, and each client downloads the fresh global model halfway through its training to catch up with what changed meanwhile. Based on Liu et al., *FedASMU: Efficient Asynchronous Federated Learning with Dynamic Staleness-aware Model Update* (AAAI 2024).

**How it works**

1. **Server.** An arriving model is mixed in right away: `global = (1 - alpha) * global + alpha * local`. Updates more than `max_staleness` versions old are discarded.
2. The weight is `alpha = mu * xi / (1 + mu * xi)` with `xi = lam / (sqrt(t) * staleness ** sigma) + iota`: smaller for stale updates and, through `sqrt(t)`, smaller as training goes on.
3. `lam`, `sigma` and `iota` are learned **per client**: when a client sends its next update, the server checks whether the weight it gave that client last time helped (did the client keep training in the same direction?) and takes one gradient step on them.
4. **Device.** Halfway through its local training, a client downloads the fresh global model (pausing for the download time) and mixes it in: `local = (1 - beta) * local + beta * fresh`, with `beta` computed and learned in the same way from `gamma` and `upsilon`.

> **Note:** compared with plain FedAsync (`fedasync.ipynb`), the learned, decaying weight keeps one stale model from dragging the global model back, and the mid-training refresh shortens how out of date a client's work is when it arrives.

> **Caution:** simplifications compared with the paper:
> * The moment to fetch the fresh model is fixed at half of the local epochs (after epoch 1 of 2). The paper chooses it with reinforcement learning (a pre-trained LSTM plus Q-learning per device), which it does not specify fully; with 2 local epochs there is only one useful moment anyway.
> * `mu_alpha`, `mu_beta` and the starting values of the control parameters are not given in the paper. The starting values are chosen so the first weights are close to FedAsync's; `mu_alpha` is chosen on validation. The learning rates of the control parameters (0.001) and `max_staleness` (99) are the paper's.
> * With those learning rates the control parameters move only a little here: mostly `iota` grows (to about 0.01 to 0.03), which keeps a minimum weight for late updates; `gamma` and `upsilon` stay at their starting values.
> * All clients train continuously, like the other approaches. The paper starts a few devices every period, with at most 10% of them training at once.
> * The mid-training refresh only matters when a client's training spans many global updates. Here it changes the result little (92.6% with it, 92.7% without, with `mu_alpha = 1`) and costs each client one extra download.

## 1. Load the simulator

Gives this notebook everything from `extended_simulator.ipynb`: all of `simulator.ipynb` (settings, `run`, ...) plus the mid-training hook `on_local_epoch` the device-side update needs.

In [1]:
%run ../extended_simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
import math

class FedASMU(Approach):
    name = "fedasmu"
    label = "FedASMU staleness-aware update"          # shown in plot legends
    copy_model_at = "dispatch"    # a client starts from the model its own previous update produced
    max_staleness = 99            # updates staler than this are discarded (tau in the paper)
    # server: weight of an arriving model, alpha = mu*xi / (1 + mu*xi), xi = lam / (sqrt(t) * staleness^sigma) + iota
    mu_alpha = 3.0                # not given in the paper; chosen on validation (1: 90.8%, 3: 92.8%, 10: 92.2%)
    lam0, sigma0, iota0 = 1.0, 0.5, 0.0           # starting control parameters (learned per client)
    lr_lam, lr_sigma, lr_iota = 0.001, 0.001, 0.001   # their learning rates (paper, Table 5)
    # device: weight of the fresh global model mixed in halfway, beta = mu*phi / (1 + mu*phi)
    fetch_fresh = True            # clients fetch the fresh global model halfway through local training
    mu_beta = 1.0                 # not given in the paper
    gamma0, upsilon0 = 1.0, 0.5   # starting control parameters (learned per client)
    lr_gamma, lr_upsilon = 0.001, 0.001

    def start(self, server):
        n = len(server.clients)
        self.ctrl = [dict(lam=self.lam0, sigma=self.sigma0, iota=self.iota0) for _ in range(n)]
        self.dev = [dict(gamma=self.gamma0, upsilon=self.upsilon0) for _ in range(n)]
        self.last = [None] * n                     # what each client's previous aggregation looked like
        super().start(server)

    # ---------- server: dynamic staleness-aware aggregation ----------
    def alpha(self, c, t, s):
        xi = max(c["lam"] / (math.sqrt(t) * s ** c["sigma"]) + c["iota"], 0.0)
        return self.mu_alpha * xi / (1 + self.mu_alpha * xi), xi

    def on_update(self, server, update):
        i, s = update.client_id, update.staleness + 1        # staleness as in the paper: t - o + 1
        if s > self.max_staleness:
            return                                           # too stale: discard
        self.learn_server(server, update)
        t = max(server.version, 1)
        a, _ = self.alpha(self.ctrl[i], t, s)
        new = average_states([server.model, update.state], [1 - a, a])
        self.last[i] = dict(upload=update.state, before=server.model, after=new, version=server.version + 1, t=t, s=s)
        server.set_model(new, [update])

    def learn_server(self, server, update):
        """One gradient step on this client's lam, sigma, iota (paper, Formula 3 and Appendix)."""
        i, prev = update.client_id, self.last[update.client_id]
        if prev is None or update.trained_version != prev["version"]:
            return                                           # needs the model its own last update produced
        # gradient of the loss at that model, estimated from how the client trained away from it
        steps = max(update.batches, 1) * server.S.lr
        g = {k: (update.start_state[k] - update.state[k]) / steps for k in update.state if torch.is_floating_point(update.state[k])}
        dloss_dalpha = sum(float((g[k] * (prev["upload"][k] - prev["before"][k])).sum()) for k in g)
        c = self.ctrl[i]
        _, xi = self.alpha(c, prev["t"], prev["s"])
        dalpha_dxi = self.mu_alpha / (1 + self.mu_alpha * xi) ** 2
        base = 1 / (math.sqrt(prev["t"]) * prev["s"] ** c["sigma"])
        grads = dict(lam=base, sigma=-c["lam"] * base * math.log(prev["s"]), iota=1.0)
        for k, lr in (("lam", self.lr_lam), ("sigma", self.lr_sigma), ("iota", self.lr_iota)):
            c[k] -= lr * dloss_dalpha * dalpha_dxi * grads[k]
        c["lam"], c["sigma"], c["iota"] = max(c["lam"], 0.0), max(c["sigma"], 0.0), max(c["iota"], 0.0)

    # ---------- device: adaptive local model adjustment ----------
    def on_local_epoch(self, server, client):
        if not self.fetch_fresh or client.epoch != max(self.local_epochs(server, client) // 2, 1):
            return
        g, o = server.version, client.version
        if g <= o:
            return                                           # no newer global model yet
        d = self.dev[client.id]
        phi = max(d["gamma"] / math.sqrt(g) * (1 - d["upsilon"] / math.sqrt(g - o + 1)), 0.0)
        beta = self.mu_beta * phi / (1 + self.mu_beta * phi)
        local = {k: v.detach().clone() for k, v in client.model.state_dict().items()}
        client.model.load_state_dict(average_states([local, server.model], [1 - beta, beta]))
        client.pause(client.download_latency)                # downloading the fresh model takes time
        # one gradient step on gamma, upsilon with the gradient on the next mini-batch (paper, Formula 9)
        grad = client.batch_gradient()
        gap = sum(float((grad[k] * (server.model[k] - local[k])).sum()) for k in grad)
        dbeta_dphi = self.mu_beta / (1 + self.mu_beta * phi) ** 2
        dphi_dgamma = (1 - d["upsilon"] / math.sqrt(g - o + 1)) / math.sqrt(g)
        dphi_dupsilon = -d["gamma"] / (math.sqrt(g) * math.sqrt(g - o + 1))
        d["gamma"] = max(d["gamma"] - self.lr_gamma * gap * dbeta_dphi * dphi_dgamma, 0.0)
        d["upsilon"] = max(d["upsilon"] - self.lr_upsilon * gap * dbeta_dphi * dphi_dupsilon, 0.0)


def average_states(models, weights):
    """Weighted average of full models (state dicts)."""
    out = {}
    for k, ref in models[0].items():
        out[k] = ref.clone() if not torch.is_floating_point(ref) else sum(m[k] * w for m, w in zip(models, weights))
    return out

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedASMU(fetch_fresh=False))` for the server-side update only, or `run(FedASMU(mu_alpha=1.0))`. It is saved as a separate run.

In [3]:
results = run(FedASMU())

fedasmu                      seed=42   val= 92.86%  acc= 93.80%  time=   593.7s  updates=  100  worst10%= 92.41%  wall=8.7s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
51,42,593.6602,100,0.938,0.1987,0.9241,9.0
